# Part 2 — Baseline FP16 Evaluation

Load **Llama-2-7B-chat at FP16** via vLLM and measure refusal rate on the first
`N=25` harmful JBB goals. Keyword-based refusal check (same family as JBB's
StringClassifier rejections) — used as a sanity baseline against the defenses
in Part 6 / Part 7.

**Outputs**: `results_part2/summary.csv`, `results_part2/raw.json`.


In [1]:
import sys
sys.path.insert(0, '.')

from jbb_common import (
    setup_vllm_env, patch_jbb, hf_login, load_jbb_dataset,
    judge_metrics, refusal_keyword_check,
    format_llama2_chat, format_llama3_chat,
    JBB_SYSTEM_PROMPT, REFUSAL_KEYWORDS,
)
setup_vllm_env()
patch_jbb()
hf_login()
goals, benign_goals, judge_df = load_jbb_dataset()
print(f"Loaded {len(goals)} harmful goals, {len(benign_goals)} benign goals, "
      f"{len(judge_df)} judge rows")


/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating harmful split: 0 examples [00:00, ? examples/s]

Generating harmful split: 100 examples [00:00, 6504.01 examples/s]

Generating benign split: 0 examples [00:00, ? examples/s]

Generating benign split: 100 examples [00:00, 20225.21 examples/s]

...

Generating benign split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating benign split: 100%|██████████| 100/100 [00:00<00:00, 22252.13 examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 300/300 [00:00<00:00, 14894.55 examples/s]

Loaded 100 harmful goals, 100 benign goals, 300 judge rows


In [2]:
import os, json, gc
import torch
import pandas as pd
from vllm import LLM, SamplingParams

llm_fp16 = LLM(
    model="meta-llama/Llama-2-7b-chat-hf",
    dtype="float16",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    tensor_parallel_size=1,
    enforce_eager=True,
)

N = 25
sampling_params = SamplingParams(temperature=0.7, max_tokens=256)

outputs = llm_fp16.generate(list(goals[:N]), sampling_params)
df_fp16 = pd.DataFrame([
    {"prompt": p, "response": o.outputs[0].text}
    for p, o in zip(goals[:N], outputs)
])

df_fp16["refused"] = df_fp16["response"].apply(refusal_keyword_check)
refusal_fp16 = df_fp16["refused"].mean()
print(f"FP16 Refusal Rate: {refusal_fp16:.2%}  (N={N})")

del llm_fp16
gc.collect()
torch.cuda.empty_cache()
print(f"Free GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


INFO 06-04 14:09:04 [utils.py:261] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'meta-llama/Llama-2-7b-chat-hf'}


INFO 06-04 14:09:17 [model.py:541] Resolved architecture: LlamaForCausalLM


INFO 06-04 14:09:17 [model.py:1561] Using max model len 4096


2026-06-04 14:09:17,453	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 06-04 14:09:17 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

Processed prompts: 100%|██████████| 25/25 [00:10<00:00,  1.17it/s, est. speed input: 46.59 toks/s, output: 590.59 toks/s]

Processed prompts: 100%|██████████| 25/25 [00:10<00:00,  2.38it/s, est. speed input: 46.59 toks/s, output: 590.59 toks/s]

[rank0]:[W604 14:10:11.708438011 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


FP16 Refusal Rate: 72.00%  (N=25)


Free GPU memory: 25.1 GB


In [3]:
os.makedirs("results_part2", exist_ok=True)

summary_df = pd.DataFrame([{
    "Model":        "Llama-2-7B-chat (FP16)",
    "N":            N,
    "Refusal Rate": round(float(refusal_fp16), 4),
}])
summary_df.to_csv("results_part2/summary.csv", index=False)

with open("results_part2/raw.json", "w") as f:
    json.dump({
        "model":        "meta-llama/Llama-2-7b-chat-hf",
        "dtype":        "float16",
        "N":            N,
        "refusal_rate": float(refusal_fp16),
        "rows": df_fp16.to_dict(orient="records"),
    }, f, indent=2)

print("\n### Part 2 summary ###")
print(summary_df.to_string(index=False))



### Part 2 summary ###
                 Model  N  Refusal Rate
Llama-2-7B-chat (FP16) 25          0.72
